In [7]:
pip install pandas faker numpy


Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()


# store data

In [3]:
store_data = []

for i in range(1, 11):
    store_data.append({
        "store_id": f"S{i:03}",
        "store_name": f"Zudio_Store_{i}",
        "city": fake.city(),
        "state": fake.state(),
        "store_size_sqft": random.randint(2000, 8000),
        "store_type": random.choice(["Mall", "High Street"]),
        "opening_date": fake.date_between(start_date='-5y', end_date='today')
    })

store_df = pd.DataFrame(store_data)


# customer data

In [4]:
customer_data = []

for i in range(1, 2001):
    customer_data.append({
        "customer_id": f"C{i:05}",
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "gender": random.choice(["Male", "Female"]),
        "date_of_birth": fake.date_of_birth(minimum_age=18, maximum_age=60),
        "phone_number": fake.phone_number(),
        "email": fake.email(),
        "city": fake.city()
    })

customer_df = pd.DataFrame(customer_data)


# product master

In [5]:
categories = ["Men", "Women", "Kids"]
brands = ["Zudio", "Trends", "Pantaloons"]

product_data = []

for i in range(1, 201):
    product_data.append({
        "product_id": f"P{i:04}",
        "product_name": fake.word().capitalize(),
        "category": random.choice(categories),
        "brand": random.choice(brands),
        "season": random.choice(["Summer", "Winter", "Festive"]),
        "gender_target": random.choice(["Men", "Women", "Unisex"]),
        "launch_date": fake.date_between(start_date='-2y', end_date='today')
    })

product_df = pd.DataFrame(product_data)


# product variants

In [6]:
sizes = ["S", "M", "L", "XL"]
colors = ["Red", "Blue", "Black", "White", "Green"]

variant_data = []

variant_id = 1

for product in product_df["product_id"]:
    for _ in range(random.randint(2, 5)):
        variant_data.append({
            "variant_id": f"V{variant_id:05}",
            "product_id": product,
            "size": random.choice(sizes),
            "colour": random.choice(colors),
            "MRP": random.randint(300, 2000),
            "cost_price": random.randint(150, 1000)
        })
        variant_id += 1

variant_df = pd.DataFrame(variant_data)


# employee data 

In [9]:
employee_data = []

emp_id = 1

for store in store_df["store_id"]:
    for _ in range(random.randint(5, 10)):
        employee_data.append({
            "employee_id": f"E{emp_id:04}",
            "employee_name": fake.name(),
            "designation": random.choice(["Sales Associate", "Manager"]),
            "store_id": store,
            "hire_date": fake.date_between(start_date='-3y', end_date='today'),
            "salary": random.randint(20000, 60000)
        })
        emp_id += 1

employee_df = pd.DataFrame(employee_data)


# inventory data

In [10]:
inventory_data = []

stock_id = 1

for store in store_df["store_id"]:
    for variant in random.sample(list(variant_df["variant_id"]), 100):
        inventory_data.append({
            "stock_id": f"ST{stock_id:05}",
            "store_id": store,
            "variant_id": variant,
            "quantity_available": random.randint(10, 150),
            "reorder_level": random.randint(5, 20),
            "last_updated": fake.date_time_this_year()
        })
        stock_id += 1

inventory_df = pd.DataFrame(inventory_data)


# sales invoice data 

In [13]:
invoice_data = []

store_ids = store_df["store_id"].tolist()
customer_ids = customer_df["customer_id"].tolist()

for i in range(1, 5001):

    store = random.choice(store_ids)

    # Filter employees for the selected store
    store_employees = employee_df.loc[
        employee_df["store_id"] == store, "employee_id"
    ].tolist()

    # Safety check (VERY IMPORTANT)
    if not store_employees:
        continue   # skip this invoice if no employee exists

    employee = random.choice(store_employees)

    invoice_data.append({
        "invoice_id": f"INV{i:06d}",
        "store_id": store,
        "customer_id": random.choice(customer_ids),
        "employee_id": employee,
        "invoice_datetime": fake.date_time_this_year(),
        "gross_amount": 0,
        "discount_amount": 0,
        "net_amount": 0
    })

invoice_df = pd.DataFrame(invoice_data)



In [15]:
invoice_df = pd.DataFrame(invoice_data)

invoice_df["gross_amount"] = invoice_df["gross_amount"].astype(float)
invoice_df["discount_amount"] = invoice_df["discount_amount"].astype(float)
invoice_df["net_amount"] = invoice_df["net_amount"].astype(float)


# sales detail data 

In [16]:

sales_detail_data = []

detail_id = 1

for invoice in invoice_df["invoice_id"]:
    total = 0
    discount_total = 0

    for _ in range(random.randint(1, 4)):
        variant = random.choice(variant_df["variant_id"])
        price = variant_df.loc[variant_df["variant_id"] == variant, "MRP"].values[0]

        qty = random.randint(1, 3)
        discount = random.uniform(0, 0.2) * price * qty

        sales_detail_data.append({
            "invoice_detail_id": f"D{detail_id:06}",
            "invoice_id": invoice,
            "variant_id": variant,
            "quantity": qty,
            "selling_price": price,
            "item_discount": round(discount, 2)
        })

        total += price * qty
        discount_total += discount
        detail_id += 1

    invoice_df.loc[invoice_df["invoice_id"] == invoice, "gross_amount"] = total
    invoice_df.loc[invoice_df["invoice_id"] == invoice, "discount_amount"] = discount_total
    invoice_df.loc[invoice_df["invoice_id"] == invoice, "net_amount"] = total - discount_total

sales_detail_df = pd.DataFrame(sales_detail_data)


 # payment data 

In [17]:
payment_data = []

for invoice in invoice_df.itertuples():
    payment_data.append({
        "payment_id": f"PAY{invoice.Index:06}",
        "invoice_id": invoice.invoice_id,
        "payment_mode": random.choice(["UPI", "Card", "Cash"]),
        "payment_date": invoice.invoice_datetime,
        "amount_paid": invoice.net_amount
    })

payment_df = pd.DataFrame(payment_data)


# export to csv 

In [18]:
store_df.to_csv("store.csv", index=False)
customer_df.to_csv("customer.csv", index=False)
product_df.to_csv("product_master.csv", index=False)
variant_df.to_csv("product_variant.csv", index=False)
employee_df.to_csv("employee.csv", index=False)
inventory_df.to_csv("inventory_stock.csv", index=False)
invoice_df.to_csv("sales_invoice.csv", index=False)
sales_detail_df.to_csv("sales_details.csv", index=False)
payment_df.to_csv("payment.csv", index=False)


In [ ]:
cd path/to/Retail-Sales-Analytics